**Code for training semantic segmentation models using the SMP library**

Import Libraries

In [1]:
import os
import numpy as np
import pandas as pd

from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.colors import BoundaryNorm

import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torch.optim import lr_scheduler

import pytorch_lightning as pl
import segmentation_models_pytorch as smp

Hyperparameters

In [2]:
MODEL_NAME = "deeplabv3plus" # Available options are: ['unet', 'unetplusplus', 'manet', 'linknet', 'fpn', 'pspnet', 'deeplabv3', 'deeplabv3plus', 'pan', 'upernet', 'segformer', 'dpt']"
ENCODER_NAME = "resnext50_32x4d" 
DEVICE_ID = 2
EPOCHS = 300
LEARNING_RATE = 3e-4
BATCH_SIZE = 128
IMAGE_SIZE = 320
SEED = 42

In [3]:
def read_part_labels(path):
    with open(path) as f:
        labels = [l.strip() for l in f if l.strip()]
        return labels

def find_part_masks(image_path, class_to_id, mask_dir, labels):
    """Return (class_id, {part_name: mask_path}) or None"""
    image_name = os.path.splitext(os.path.basename(image_path))[0]
    class_name = os.path.basename(os.path.dirname(image_path))
    class_id = class_to_id[class_name]
    class_mask_dir = os.path.join(mask_dir, str(class_id))
    if not os.path.isdir(class_mask_dir):
        return None

    masks = {}
    for f in os.listdir(class_mask_dir):
        if image_name in f:
            for lbl in labels:
                if lbl in f:
                    masks[lbl] = os.path.join(class_mask_dir, f)

    return (class_id, masks) if masks else None


def visualize(*, labels=None, **images):
    """Plot images + masks in one row with perfect discrete colormap legend (unlimited classes)"""
    n = len(images)
    plt.figure(figsize=(4 + 4 * n, 6))  # auto-scale width
    for i, (name, data) in enumerate(images.items(), 1):
        plt.subplot(1, n, i)
        plt.xticks([]); plt.yticks([])
        plt.title(name.replace("_", " ").title(), fontsize=14)
        arr = np.asarray(data)

        # ---------------- IMAGE ----------------
        if name == "image":
            # Handle CHW → HWC
            if arr.ndim == 3 and arr.shape[0] == 3:
                arr = arr.transpose(1, 2, 0)
            # Normalize to 0–1 if needed
            if arr.max() > 1.0:
                arr = arr / 255.0
            plt.imshow(arr)
            continue

        # ---------------- MASK PLOTTING ----------------
        # Supports unlimited integer classes with exact color mapping
        mask = arr.squeeze()  # H x W
        cmap = plt.get_cmap("turbo")  # # Alternatives: "nipy_spectral", "hsv", "jet", "gist_ncar"        
        bounds = np.arange(-0.5, mask.max() + 1.5, 1)
        norm = BoundaryNorm(bounds, cmap.N)
        plt.imshow(mask, cmap=cmap, norm=norm, interpolation="nearest")

        # ---------------- LEGEND ----------------
        # Exact color match between image and legend
        handles = [
            Patch(
                color=cmap(norm(v)),  # exact color used in the image
                label=labels[v] if v < len(labels) else f"Class {v} (unknown)"
            )
            for v in np.unique(mask)
        ]

        plt.legend(
            handles=handles, title="Classes", bbox_to_anchor=(1.05, 1),
            loc="upper left", borderaxespad=0.0, fontsize=10, title_fontsize=12
        )

    plt.tight_layout()
    plt.show()


In [4]:
class PartWholeDataset(Dataset):
    def __init__(self, dataset_path, image_size=320):
        self.image_size = image_size

        # -------- Paths --------
        self.img_dir = os.path.join(dataset_path, "images")
        self.mask_dir = os.path.join(dataset_path, "AnnotationMasksPerclass")

        # -------- Labels --------
        self.labels = ["background"] + read_part_labels(
            os.path.join(dataset_path, "part_labels.txt")
        )
        self.num_parts = len(self.labels)

        # -------- Classes --------
        classes = pd.read_csv(
            os.path.join(dataset_path, "classes.txt"),
            sep=" ", header=None, names=["id", "name"]
        )
        self.class_to_id = {
            name: idx + 1 for idx, name in enumerate(classes["name"])
        }

        # -------- Images --------
        images = pd.read_csv(
            os.path.join(dataset_path, "images.txt"),
            sep=" ", header=None, names=["id", "name"]
        )

        # Build image → (class_id, mask_dict) mapping ONCE
        self.samples = []
        for name in images["name"]:
            img_path = os.path.join(self.img_dir, name)
            info = find_part_masks(
                img_path, self.class_to_id, self.mask_dir, self.labels
            )
            if info:
                self.samples.append((img_path, *info))

        # -------- Transforms --------
        self.img_transform = transforms.Compose([
            transforms.Resize(
                (image_size, image_size),
                interpolation=transforms.InterpolationMode.BILINEAR
            ),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, class_id, mask_dict = self.samples[idx]

        # Load image
        img = Image.open(img_path).convert("RGB")
        img = self.img_transform(img)

        # Combined mask
        mask = np.zeros((self.image_size, self.image_size), dtype=np.int64)
        for label_idx, label in enumerate(self.labels):
            path = mask_dict.get(label)
            if not path:
                continue

            pm = Image.open(path).convert("L")
            pm = pm.resize((self.image_size, self.image_size), Image.NEAREST)
            mask[np.array(pm) > 127] = label_idx
        mask = torch.from_numpy(mask)
        
        return img, mask, class_id


In [5]:
# Create Dataset
IMAGE_SIZE = 320
dataset_path = r'/home/c/choton/beemachine/datasets/Others/CUB_200_2011/'
full_dataset = PartWholeDataset(image_size=IMAGE_SIZE, dataset_path=dataset_path)
labels = full_dataset.labels
torch.manual_seed(SEED)

# Train / Val / Test Split (75:15:10)
n_total = len(full_dataset)
n_train = int(0.75 * n_total)
n_val = int(0.15 * n_total)
n_test = n_total - n_train - n_val

train_set, val_set, test_set = random_split(full_dataset, [n_train, n_val, n_test])

# DataLoaders
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
print(f"Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}")
num_classes = len(labels)
print(f"Training with number of part labels (including background) = {num_classes}")
print("Part labels:", labels)

Train: 1416, Val: 283, Test: 189
Training with number of part labels (including background) = 12
Part labels: ['background', 'body', 'head', 'neck', 'beak', 'tail', 'right_wing', 'left_wing', 'right_leg', 'left_leg', 'right_eye', 'left_eye']


In [6]:
class CamVidModel(pl.LightningModule):
    def __init__(self, arch, encoder_name, in_channels, out_classes, **kwargs):
        super().__init__()
        self.model = smp.create_model(
            arch,
            encoder_name=encoder_name,
            in_channels=in_channels,
            classes=out_classes,
            **kwargs,
        )

        # Preprocessing parameters for image normalization
        params = smp.encoders.get_preprocessing_params(encoder_name)
        self.number_of_classes = out_classes
        self.register_buffer("std", torch.tensor(params["std"]).view(1, 3, 1, 1))
        self.register_buffer("mean", torch.tensor(params["mean"]).view(1, 3, 1, 1))

        # Loss function for multi-class segmentation
        self.loss_fn = smp.losses.DiceLoss(smp.losses.MULTICLASS_MODE, from_logits=True)

        # Step metrics tracking
        self.training_step_outputs = []
        self.validation_step_outputs = []
        self.test_step_outputs = []

    def forward(self, image):
        # Normalize image
        image = (image - self.mean) / self.std
        mask = self.model(image)
        return mask

    def shared_step(self, batch, stage):
        image, mask, _ = batch

        # Ensure that image dimensions are correct
        assert image.ndim == 4  # [batch_size, channels, H, W]

        # Ensure the mask is a long (index) tensor
        mask = mask.long()

        # Mask shape
        assert mask.ndim == 3  # [batch_size, H, W]

        # Predict mask logits
        logits_mask = self.forward(image)

        assert (
            logits_mask.shape[1] == self.number_of_classes
        )  # [batch_size, number_of_classes, H, W]

        # Ensure the logits mask is contiguous
        logits_mask = logits_mask.contiguous()

        # Compute loss using multi-class Dice loss (pass original mask, not one-hot encoded)
        loss = self.loss_fn(logits_mask, mask)

        # Apply softmax to get probabilities for multi-class segmentation
        prob_mask = logits_mask.softmax(dim=1)

        # Convert probabilities to predicted class labels
        pred_mask = prob_mask.argmax(dim=1)

        # Compute true positives, false positives, false negatives, and true negatives
        tp, fp, fn, tn = smp.metrics.get_stats(
            pred_mask, mask, mode="multiclass", num_classes=self.number_of_classes
        )

        return {
            "loss": loss,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "tn": tn,
        }

    def shared_epoch_end(self, outputs, stage):
        # Aggregate step metrics
        tp = torch.cat([x["tp"] for x in outputs])
        fp = torch.cat([x["fp"] for x in outputs])
        fn = torch.cat([x["fn"] for x in outputs])
        tn = torch.cat([x["tn"] for x in outputs])

        # Per-image IoU and dataset IoU calculations
        per_image_iou = smp.metrics.iou_score(
            tp, fp, fn, tn, reduction="micro-imagewise"
        )
        dataset_iou = smp.metrics.iou_score(tp, fp, fn, tn, reduction="micro")

        metrics = {
            f"{stage}_per_image_iou": per_image_iou,
            f"{stage}_dataset_iou": dataset_iou,
        }

        self.log_dict(metrics, prog_bar=True)

    def training_step(self, batch, batch_idx):
        train_loss_info = self.shared_step(batch, "train")
        self.training_step_outputs.append(train_loss_info)
        return train_loss_info

    def on_train_epoch_end(self):
        self.shared_epoch_end(self.training_step_outputs, "train")
        self.training_step_outputs.clear()

    def validation_step(self, batch, batch_idx):
        valid_loss_info = self.shared_step(batch, "valid")
        self.validation_step_outputs.append(valid_loss_info)
        return valid_loss_info

    def on_validation_epoch_end(self):
        self.shared_epoch_end(self.validation_step_outputs, "valid")
        self.validation_step_outputs.clear()

    def test_step(self, batch, batch_idx):
        test_loss_info = self.shared_step(batch, "test")
        self.test_step_outputs.append(test_loss_info)
        return test_loss_info

    def on_test_epoch_end(self):
        self.shared_epoch_end(self.test_step_outputs, "test")
        self.test_step_outputs.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=2e-4)
        scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-5)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch",
                "frequency": 1,
            },
        }

In [7]:
model = CamVidModel(MODEL_NAME, ENCODER_NAME, in_channels=3, out_classes=num_classes)

In [8]:
trainer = pl.Trainer(max_epochs=EPOCHS, log_every_n_steps=1, devices=[0,1,2,3,4,5,6,7])

trainer.fit(
    model,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/c/choton/miniconda3/envs/beemachine_new/lib/python3.11/site-packages/torch/__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  return _C._get_float32_matmul_precision()


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/c/choton/miniconda3/envs/beemachine_new/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/logger_connector/result.py:434: It is recommended to use `self.log('valid_per_image_iou', ..., sync_dist=True)` when logging on epoch level in distributed setting to accumulate the metric across devices.
/home/c/choton/miniconda3/envs/beemachine_new/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/logger_connector/result.py:434: It is recommended to use `self.log('valid_dataset_iou', ..., sync_dist=True)` when logging on epoch level in distributed setting to accumulate the metric across devices.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=300` reached.
